# Steerling-8B: Concept Unlearning

**Concept unlearning** suppresses a target concept during generation, with no weight updates. Steerling does this at inference time by intervening on its concept representations.

The public unlearning recipe is `injection_relu`, which combines two mechanisms. **Negative injection** adds the negated concept direction to the residual stream (the running hidden state) at the deeper layers, pushing generation away from the concept. The **ReLU logit mask** then subtracts the concept's positive alignment from the output logits, removing the specific tokens the concept promotes without boosting unrelated ones.

This notebook generates twice on the same prompt, a reference and an unlearned run, then checks that the target concept's activation drops.

**Requirements:** an interpretable Steerling checkpoint and a GPU with >= 18 GB VRAM.

## 1. Load the model

Unlearning needs the **interpretable** model, which exposes the concept heads the intervention acts on. We load via HuggingFace `AutoModel` and wrap it in `SteerlingGenerator`.

In [13]:
import sys
sys.path.insert(0, "/fss/nguyeng/june-release/steerling")

import torch
from steerling import SteerlingGenerator, GenerationConfig, SteeringConfig
from steerling.configs.causal_diffusion import CausalDiffusionConfig
from steerling.configs.concept import ConceptConfig
from steerling.data.tokenizer import SteerlingTokenizer
from steerling.models.interpretable.interpretable_causal_diffusion import InterpretableCausalDiffusionLM
from steerling.inference.checkpoint_utils import load_config, load_state_dict

model_id = "guidelabs/steerling-8b"

raw = load_config(model_id)
model_cfg = CausalDiffusionConfig.model_validate(
    {k: v for k, v in raw.items() if k in CausalDiffusionConfig.model_fields}
)
concept_fields = set(ConceptConfig.model_fields)
concept_cfg = ConceptConfig.model_validate({k: v for k, v in raw.items() if k in concept_fields})
vocab_size = raw.get("vocab_size")

model = InterpretableCausalDiffusionLM(model_cfg, concept_cfg, vocab_size)
state = load_state_dict(model_id)
model.load_state_dict(state, strict=False)
if hasattr(model, "transformer"):
    model.transformer._restore_weight_tying()
model = model.to(dtype=torch.bfloat16)

tokenizer = SteerlingTokenizer(instruct=raw.get("instruct", False))
generator = SteerlingGenerator(
    model=model,
    tokenizer=tokenizer,
    model_config=model_cfg,          # the CausalDiffusionConfig you built above
    is_interpretable=True,
    device="cuda",
)

# flex attention needs to be wired, which from_pretrained/from_model normally do:
import steerling.models.layers.causal_diffusion_layers as layers
from torch.nn.attention.flex_attention import flex_attention
layers.compiled_flex_attention = flex_attention

print(type(generator.model).__name__)              # InterpretableCausalDiffusionLM
print(hasattr(generator.model, "_forward_with_injection"))  # True
assert generator.is_interpretable

InterpretableCausalDiffusionLM
True


## 2. Choose a concept and prime the prompt

Set the target concept and a prompt. To make suppression visible, we **prime** the prompt toward the concept, the same way the steering evaluation does: prepend a short description so an unsteered model would lean into the concept. Unlearning should pull the generation back out.

Set `CONCEPT_ID`, `CONCEPT_LABEL`, and `CONCEPT_DESCRIPTION` to a concept from your concept table.

In [14]:
# --- target concept (replace with one from your concept table) ---
CONCEPT_ID = 1299
CONCEPT_LABEL = "Book Publishing Industry"
CONCEPT_DESCRIPTION = "The 'publish' morpheme in many surface variants (publish, publisher, publication, Publishing, .pub, _publish), industry infrastructure (imprint, ISBN, Kindle, ePub, Penguin, Harper, Barnes & Noble, Ingram..."

# --- prompt, primed toward the concept ---
BASE_PROMPT = "Tell me about your weekend."
PRIMED_PROMPT = f"We are discussing about {CONCEPT_LABEL}: {CONCEPT_DESCRIPTION}\n{BASE_PROMPT}"

# --- generation budget ---
MAX_NEW_TOKENS = 128
STEPS = 128

# --- unlearning strength ---
# mai_lm_target is the raw injection alpha. Negative suppresses.
# -7 is the preset default; sweeps found ~-5 best for grouped concepts.
ALPHA = -7.0
# inject_layer=None resolves to n_layers // 2 inside the generator.
INJECT_LAYER = None

print(PRIMED_PROMPT)

We are discussing about Book Publishing Industry: The 'publish' morpheme in many surface variants (publish, publisher, publication, Publishing, .pub, _publish), industry infrastructure (imprint, ISBN, Kindle, ePub, Penguin, Harper, Barnes & Noble, Ingram...
Tell me about your weekend.


## 3. Reference generation

First generate with no intervention. This is the baseline the unlearned run is compared against. Reference sampling uses `temperature=1.2`, `top_p=0.8`, `repetition_penalty=1.1`.

In [15]:
ref_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=1.2,
    top_p=0.8,
    repetition_penalty=1.1,
    seed=0,
)

reference = generator.generate_full(PRIMED_PROMPT, ref_config)
print(reference.text)

 What is a book? What is an imprint? What is an ISBN? What is a genre? What is an author? What is a publisher? What is a writer? What is a title? What is a chapter? What is a sentence? What is a paragraph? What is a page?
We will be talking about the history of the publishing industry and how it has changed over time. We'll also discuss the different types of publishers, such as traditional publishers, self-publishers, and independent presses.
How do books get published? How does the publishing process work? What are some of the biggest challenges facing publishers today?


## 4. Unlearn the concept (`injection_relu`)

`SteeringConfig.injection_relu` builds the unlearning config: negative injection of the concept direction plus the ReLU logit mask on the focal concept (fixed strength 20). Negative injection steers away from the concept, and the mask cleans up the residual tokens it would still promote. We pass the config to `generate_steered`. Steered sampling uses `temperature=0.9`, `top_p=0.7`, `repetition_penalty=1.5`.

In [16]:
steered_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=0.9,
    top_p=0.7,
    repetition_penalty=1.5,
    seed=0,
)

unlearn = SteeringConfig.injection_relu(
    concept_ids=CONCEPT_ID,
    mai_lm_target=ALPHA,
    inject_layer=INJECT_LAYER,
)

unlearned = generator.generate_steered(PRIMED_PROMPT, steered_config, unlearn)
print(unlearned.text)

 What did you do yesterday? Where'd ya go tonight ? What should we talk about tomorrow morning?? Here is the last word of this day that I would like to share with all my friends here on TalkTalkShop , which means talking shopping.
Good bye... Good luck out there! You don't know what happened today.... We're gonna be back again next week!

Here it goes folks..... Reviewing some of our old words and adding a lot more new ones...... It looks good so far.......... There's no room left though.......I need help!!!!!! How can i fit everything else??? That'll get everyone thinking.........
